In [0]:
%sql
/* THIS CODE IS NOT FULLY TESTED YET. It still needs to be verified, but it uses the same
 * algorithm found in Chris_Knoll's script: https://gist.github.com/chrisknoll/c820cc12d833db2e3d1e
 */

--------------------------------------------------------------------------------------------------------------
--- Adapted to Databricks SQL (OMOP v5 DOSE_ERA) from Pure SQL drug_era written by Chris_Knoll:
--- https://gist.github.com/chrisknoll/c820cc12d833db2e3d1e
--- INTERVAL set to 30 days
---
--- Chris Knoll's comments are after two dashes
--- Taylor Delehanty's comments are after three dashes
--- Proper schema name needs to replace "<schema>" in the code
--- Proper schema name needs to replace "<vocabulary_schema>" if vocabularies are separate
--- Works with dose_era_id being self-generated (IDENTITY column)
--------------------------------------------------------------------------------------------------------------

TRUNCATE TABLE _exponent.omop.dose_era;

WITH cteDrugTarget AS (
    SELECT
        d.drug_exposure_id,
        d.person_id,
        c.concept_id AS drug_concept_id,                    --- Ingredient-level concept
        d.dose_unit_concept_id AS unit_concept_id,
        d.effective_drug_dose AS dose_value,
        d.drug_exposure_start_date,
        d.days_supply,

        --- Determine exposure end date
        --- 1. Use drug_exposure_end_date if present
        --- 2. Else use start_date + days_supply (if valid)
        --- 3. Else default to start_date + 1 day
        COALESCE(
            d.drug_exposure_end_date,
            CASE
                WHEN d.days_supply IS NOT NULL AND d.days_supply > 0
                    THEN date_add(d.drug_exposure_start_date, CAST(d.days_supply AS INT))
                ELSE NULL
            END,
            date_add(d.drug_exposure_start_date, 1)
        ) AS drug_exposure_end_date

    FROM _exponent.omop.drug_exposure d
         JOIN _exponent.omop.concept_ancestor ca
           ON ca.descendant_concept_id = d.drug_concept_id
         JOIN _exponent.omop.concept c
           ON ca.ancestor_concept_id = c.concept_id

    --- Keep only ingredient-level RxNorm concepts
    WHERE c.vocabulary_id = 'RxNorm'
      AND c.concept_class_id = 'Ingredient'

    /* Depending on the needs of your data, you can put more filters on to your code.
     * We assign 0 to unmapped drug_concept_id's, and we found data where days_supply was negative.
     * We don't want different drugs put in the same era, so the code below shows how we filtered them out.
     * We also don't want negative days_supply, because that will pull our end_date before the start_date.
     * For now, we are filtering those out as well, but this is a data quality issue that we are trying to solve.
     */
     --- AND d.drug_concept_id != 0
     --- AND (d.days_supply IS NULL OR d.days_supply >= 0)
)

-----------------------------------------------------------------------------------------------------------------------------
, cteEndDates AS (
    SELECT
        person_id,
        drug_concept_id,
        unit_concept_id,
        dose_value,
        date_add(event_date, -30) AS end_date         --- subtract 30-day persistence window
    FROM
    (
        SELECT
            person_id,
            drug_concept_id,
            unit_concept_id,
            dose_value,
            event_date,
            event_type,

            --- Track most recent start ordinal
            MAX(start_ordinal) OVER (
                PARTITION BY person_id, drug_concept_id, unit_concept_id, dose_value
                ORDER BY event_date, event_type
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) AS start_ordinal,

            ROW_NUMBER() OVER (
                PARTITION BY person_id, drug_concept_id, unit_concept_id, dose_value
                ORDER BY event_date, event_type
            ) AS overall_ord

        FROM
        (
            --- Drug exposure starts
            SELECT
                person_id,
                drug_concept_id,
                unit_concept_id,
                dose_value,
                drug_exposure_start_date AS event_date,
                -1 AS event_type,
                ROW_NUMBER() OVER (
                    PARTITION BY person_id, drug_concept_id, unit_concept_id, dose_value
                    ORDER BY drug_exposure_start_date
                ) AS start_ordinal
            FROM cteDrugTarget

            UNION ALL

            --- Drug exposure ends (+30 day window)
            SELECT
                person_id,
                drug_concept_id,
                unit_concept_id,
                dose_value,
                date_add(drug_exposure_end_date, 30) AS event_date,
                1 AS event_type,
                NULL AS start_ordinal
            FROM cteDrugTarget
        ) RAWDATA
    ) e
    WHERE (2 * e.start_ordinal) - e.overall_ord = 0
)

-----------------------------------------------------------------------------------------------------------------------------
, cteDoseEraEnds AS (
    SELECT
        dt.person_id,
        dt.drug_concept_id,
        dt.unit_concept_id,
        dt.dose_value,
        dt.drug_exposure_start_date,
        MIN(e.end_date) AS dose_era_end_date

    FROM cteDrugTarget dt
    JOIN cteEndDates e
      ON dt.person_id       = e.person_id
     AND dt.drug_concept_id = e.drug_concept_id
     AND dt.unit_concept_id = e.unit_concept_id
     AND dt.dose_value      <=> e.dose_value
     AND e.end_date >= dt.drug_exposure_start_date

    GROUP BY
        dt.person_id,
        dt.drug_concept_id,
        dt.unit_concept_id,
        dt.dose_value,
        dt.drug_exposure_start_date
)

-----------------------------------------------------------------------------------------------------------------------------
--- Final DOSE_ERA insert
--- dose_era_id auto-generates via IDENTITY column
-----------------------------------------------------------------------------------------------------------------------------
INSERT INTO _exponent.omop.dose_era (
    person_id,
    drug_concept_id,
    unit_concept_id,
    dose_value,
    dose_era_start_date,
    dose_era_end_date
)
SELECT
    person_id,
    drug_concept_id,
    unit_concept_id,
    dose_value,
    MIN(drug_exposure_start_date) AS dose_era_start_date,
    dose_era_end_date
FROM cteDoseEraEnds
GROUP BY
    person_id,
    drug_concept_id,
    unit_concept_id,
    dose_value,
    dose_era_end_date
ORDER BY
    person_id,
    drug_concept_id;

In [0]:
%sql
SELECT 
*
FROM _exponent.omop.relationship